In [ ]:
# Import required libraries
import os
import zipfile
import urllib.request
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)

## Task 1

In [ ]:
# Create a random 3x3 matrix and a 3x3 filter
np.random.seed(42)

matrix = np.random.randint(0, 10, (3, 3)).astype(np.float32)
kernel = np.array([[1, 0, -1],
                   [1, 0, -1],
                   [1, 0, -1]], dtype=np.float32)

print("Input Matrix:\n", matrix)
print("\nFilter:\n", kernel)

In [ ]:
# Padding with P = 1
padded_matrix = np.pad(matrix, ((1, 1), (1, 1)), mode='constant', constant_values=0)

print("Padded Matrix:\n", padded_matrix)

In [ ]:
# Convolution using nested loops
convolution = np.zeros((3, 3), dtype=np.float32)

for i in range(3):
    for j in range(3):
        window = padded_matrix[i:i+3, j:j+3]
        convolution[i, j] = np.sum(window * kernel)

print("Convolution Output:\n", convolution)

In [ ]:
# ReLU and Sigmoid activation functions
relu_output = np.maximum(convolution, 0)
sigmoid_output = 1 / (1 + np.exp(-convolution))

print("ReLU Output:\n", relu_output)
print("\nSigmoid Output:\n", sigmoid_output)

In [ ]:
# Max Pooling and Min Pooling with a 2x2 window and stride 1
max_pool = np.zeros((2, 2), dtype=np.float32)
min_pool = np.zeros((2, 2), dtype=np.float32)

for i in range(2):
    for j in range(2):
        window = relu_output[i:i+2, j:j+2]
        max_pool[i, j] = np.max(window)
        min_pool[i, j] = np.min(window)

print("Max Pooling Output:\n", max_pool)
print("\nMin Pooling Output:\n", min_pool)

In [ ]:
# Global Average Pooling
global_average_pooling = np.mean(relu_output)
print("Global Average Pooling Output:", global_average_pooling)

In [ ]:
# Flattening and two Dense layers
flattened = relu_output.flatten()

weights1 = np.random.rand(len(flattened), 5)
bias1 = np.zeros(5)
dense1 = np.dot(flattened, weights1) + bias1
dense1 = np.maximum(dense1, 0)

weights2 = np.random.rand(5, 2)
bias2 = np.zeros(2)
dense2 = np.dot(dense1, weights2) + bias2

print("Flattened Output:\n", flattened)
print("\nFirst Dense Layer Output:\n", dense1)
print("\nSecond Dense Layer Output:\n", dense2)

In [ ]:
# Demonstrate Batch Normalization and Dropout
sample_data = np.random.random((8, 9)).astype(np.float32)

batch_norm = layers.BatchNormalization()
normalized_data = batch_norm(sample_data, training=True)

dropout_layer = layers.Dropout(0.25)
dropout_output = dropout_layer(normalized_data, training=True)

print("Input Batch Shape:", sample_data.shape)
print("After Batch Normalization:", normalized_data.shape)
print("After Dropout:", dropout_output.shape)

# Early stopping callback (used during training in Task 2)
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
print("Early Stopping callback is ready.")

In [ ]:
# A small CNN model showing the main layers on a 3x3 input
basic_cnn = models.Sequential()
basic_cnn.add(layers.Input(shape=(3, 3, 1)))
basic_cnn.add(layers.Conv2D(2, (3, 3), padding='same'))
basic_cnn.add(layers.BatchNormalization())
basic_cnn.add(layers.Activation('relu'))
basic_cnn.add(layers.MaxPooling2D(pool_size=(2, 2)))
basic_cnn.add(layers.Flatten())
basic_cnn.add(layers.Dense(8, activation='relu'))
basic_cnn.add(layers.Dropout(0.25))
basic_cnn.add(layers.Dense(1, activation='sigmoid'))

basic_cnn.summary()

## Task 2

In [ ]:
# Download Tiny ImageNet dataset
dataset_url = "https://cs231n.stanford.edu/tiny-imagenet-200.zip"
zip_path = "tiny-imagenet-200.zip"
dataset_path = "tiny-imagenet-200"

if not os.path.exists(zip_path):
    print("Downloading Tiny ImageNet...")
    urllib.request.urlretrieve(dataset_url, zip_path)
    print("Download completed.")

if not os.path.exists(dataset_path):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Extraction completed.")

train_dir = os.path.join(dataset_path, 'train')
print("Dataset folder:", dataset_path)
print("Train folder found:", os.path.exists(train_dir))

In [ ]:
# Select any 3 classes from Tiny ImageNet
with open(os.path.join(dataset_path, 'wnids.txt'), 'r') as file:
    all_classes = [line.strip() for line in file.readlines()]

selected_classes = all_classes[:3]

# Read readable class names from words.txt
class_words = {}
with open(os.path.join(dataset_path, 'words.txt'), 'r') as file:
    for line in file:
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            class_words[parts[0]] = parts[1]

selected_class_names = [class_words.get(class_id, class_id) for class_id in selected_classes]

print("Selected class IDs:", selected_classes)
print("Selected class names:", selected_class_names)

In [ ]:
# Load 200 RGB images from each selected class
np.random.seed(42)
images = []
labels = []

for class_index, class_id in enumerate(selected_classes):
    images_dir = os.path.join(train_dir, class_id, 'images')
    image_files = [name for name in os.listdir(images_dir) if name.lower().endswith('.jpeg')]
    np.random.shuffle(image_files)
    image_files = image_files[:200]

    for image_name in image_files:
        image_path = os.path.join(images_dir, image_name)
        image = cv2.imread(image_path)

        if image is not None:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (64, 64))
            images.append(image)
            labels.append(class_index)

X = np.array(images, dtype=np.uint8)
y = np.array(labels, dtype=np.int64)

print("Image data shape:", X.shape)
print("Label data shape:", y.shape)
print("Total images:", len(X))

for i, class_name in enumerate(selected_class_names):
    print(class_name, "-", np.sum(y == i), "images")

if len(X) != 600:
    raise ValueError("600 images were not loaded. Check the dataset folder and selected classes.")

In [ ]:
# Display one image from each selected class
plt.figure(figsize=(10, 4))

for i in range(3):
    image_index = np.where(y == i)[0][0]
    plt.subplot(1, 3, i + 1)
    plt.imshow(X[image_index])
    plt.title(selected_class_names[i])
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Split data into 70% training, 15% validation and 15% testing
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Normalize pixel values between 0 and 1
X_train = X_train.astype('float32') / 255.0
X_val = X_val.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print("Training images:", X_train.shape[0])
print("Validation images:", X_val.shape[0])
print("Testing images:", X_test.shape[0])

In [ ]:
# Build the custom CNN model
custom_cnn = models.Sequential()
custom_cnn.add(layers.Input(shape=(64, 64, 3)))

custom_cnn.add(layers.Conv2D(32, (3, 3), padding='same'))
custom_cnn.add(layers.BatchNormalization())
custom_cnn.add(layers.Activation('relu'))
custom_cnn.add(layers.MaxPooling2D(pool_size=(2, 2)))

custom_cnn.add(layers.Conv2D(32, (3, 3), padding='same'))
custom_cnn.add(layers.BatchNormalization())
custom_cnn.add(layers.Activation('relu'))
custom_cnn.add(layers.Lambda(
    lambda x: -tf.nn.max_pool2d(-x, ksize=2, strides=2, padding='VALID'),
    name='MinPooling'
))

custom_cnn.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
custom_cnn.add(layers.GlobalAveragePooling2D())
custom_cnn.add(layers.Dense(64, activation='relu'))
custom_cnn.add(layers.Dropout(0.5))
custom_cnn.add(layers.Dense(3, activation='softmax'))

custom_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

custom_cnn.summary()

In [ ]:
# Train the custom CNN
history = custom_cnn.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

In [ ]:
# Plot training and validation loss curves
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Custom CNN - Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    return accuracy, precision, recall, f1

# Predict test labels using custom CNN
custom_predictions = custom_cnn.predict(X_test)
custom_predicted_labels = np.argmax(custom_predictions, axis=1)

custom_scores = calculate_metrics(y_test, custom_predicted_labels)

results = []
results.append({
    'Model': 'Custom CNN',
    'Accuracy': custom_scores[0],
    'Precision': custom_scores[1],
    'Recall': custom_scores[2],
    'F1 Score': custom_scores[3]
})

print("Custom CNN Test Metrics")
print("Accuracy:", custom_scores[0])
print("Precision:", custom_scores[1])
print("Recall:", custom_scores[2])
print("F1 Score:", custom_scores[3])

## Task 3

In [ ]:
# Import pretrained CNN models and their preprocessing functions
from tensorflow.keras.applications import VGG16, MobileNetV2, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

In [ ]:
# Function to train and test a transfer learning model
def train_transfer_model(model_name, application_model, preprocess_function):
    # Load pretrained model without its original classification head
    base_model = application_model(
        weights='imagenet',
        include_top=False,
        input_shape=(64, 64, 3)
    )
    base_model.trainable = False

    # Add a new classification head for 3 classes
    input_layer = layers.Input(shape=(64, 64, 3))
    x = base_model(input_layer, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    output_layer = layers.Dense(3, activation='softmax')(x)

    transfer_model = models.Model(input_layer, output_layer)
    transfer_model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # Apply the preprocessing required by the selected pretrained model
    train_images = preprocess_function(X_train * 255.0)
    val_images = preprocess_function(X_val * 255.0)
    test_images = preprocess_function(X_test * 255.0)

    transfer_early_stop = EarlyStopping(
        monitor='val_loss', patience=2, restore_best_weights=True
    )

    print("Training", model_name)
    transfer_model.fit(
        train_images, y_train,
        epochs=8,
        batch_size=32,
        validation_data=(val_images, y_val),
        callbacks=[transfer_early_stop]
    )

    # Predict test labels
    predictions = transfer_model.predict(test_images)
    predicted_labels = np.argmax(predictions, axis=1)
    scores = calculate_metrics(y_test, predicted_labels)

    print(model_name, "Test Metrics")
    print("Accuracy:", scores[0])
    print("Precision:", scores[1])
    print("Recall:", scores[2])
    print("F1 Score:", scores[3])

    results.append({
        'Model': model_name,
        'Accuracy': scores[0],
        'Precision': scores[1],
        'Recall': scores[2],
        'F1 Score': scores[3]
    })

    return transfer_model

In [ ]:
# Train VGG16
vgg_model = train_transfer_model('VGG16', VGG16, vgg_preprocess)

In [ ]:
# Train MobileNetV2
mobilenet_model = train_transfer_model('MobileNetV2', MobileNetV2, mobilenet_preprocess)

In [ ]:
# Train ResNet50
resnet_model = train_transfer_model('ResNet50', ResNet50, resnet_preprocess)

In [ ]:
# Compare all four models using test metrics
results_df = pd.DataFrame(results)
print(results_df)

# Display comparison as a bar chart
results_df.plot(x='Model', y=['Accuracy', 'Precision', 'Recall', 'F1 Score'],
                kind='bar', figsize=(10, 5))
plt.title('Custom CNN and Transfer Learning Model Comparison')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()